In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("C://Users//DELL//Desktop//100DaysofML//Exploratory_Data_Analysis//train.csv")

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Removing the columns that do not contribute in prediction of target variable
df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], inplace=True)

In [5]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## Splitting dataset into training and testing data

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns='Survived'), 
                                                    df['Survived'], 
                                                    test_size=0.2, 
                                                    random_state=69)

In [8]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
507,1,male,NaN,0,0,26.55,S
201,3,male,NaN,8,2,69.55,S
180,3,female,NaN,8,2,69.55,S
78,2,male,0.83,0,2,29.00,S
72,2,male,21.00,0,0,73.50,S


## Identifying columns with missing values

In [9]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

## High level plan 

In [10]:
 #Plan for the dataset:
#     Note: All feature transformations applied with ColumnTransformer for conveinence
#     Steps for the pipeline:
#       1. Handling missing values: Impute missing values for Age(mean) and Embarked(mode) using SimpleImputer 
#       2. Handling Categorical Columns: i. Ordinal encoding for Pclass (OE), 
#                                        ii. nominal encoding for Sex and Embarked columns (OHE)
#       3. Feature Scaling: minmaxscaler()
#       4. Feature Selection
#       5. Decision Tree


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline,make_pipeline

In [12]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
507,1,male,NaN,0,0,26.55,S
201,3,male,NaN,8,2,69.55,S
180,3,female,NaN,8,2,69.55,S
78,2,male,0.83,0,2,29.00,S
72,2,male,21.00,0,0,73.50,S


### 1. Handling columns with missing values

In [13]:
tf1 = ColumnTransformer([
    ('age_imputer', SimpleImputer(), [2] ), 
    ("embarked_imputer", SimpleImputer(strategy='most_frequent'), [6])
    ], 
    remainder='passthrough')
# note: After this transformation here, Age will have index 0 and and Embarked will have index of 1.
# indexes:
# 0 --> Age
# 1 --> Embarked
# 2 --> Pclass
# 3 --> Sex
# 4 --> SibSp
# 5 --> Parch
# 6 --> Fare

### 2. Handling Categorical columns

In [14]:
tf2 = ColumnTransformer([
    ('embarked_sex_ohe', OneHotEncoder(sparse_output=False, handle_unknown='error'), [1, 3])
], remainder='passthrough')

### 3. Feature Scaling

In [15]:
tf3 = ColumnTransformer([
    ('scale_all', MinMaxScaler(), slice(0,10))
], remainder='passthrough')

### 4. Feature Selection

In [16]:
tf4 = SelectKBest(score_func=chi2,k=8)

### 5. Model Training  

In [17]:
tf5 = DecisionTreeClassifier()

## Creating pipeline

In [18]:
pipe = Pipeline([
    ("imputing", tf1),
    ("encoding", tf2),
    ("scaling", tf3),
    ("selection", tf4), 
    ("model", tf5)
])

In [19]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('imputing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('age_imputer',
                                                  SimpleImputer(), [2]),
                                                 ('embarked_imputer',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('encoding',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('embarked_sex_ohe',
                                                  OneHotEncoder(sparse_output=False),
                                                  [1, 3])])),
                ('scaling',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale_all', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('selection',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x0000025484F7FEC0>)),
                ('model', DecisionTreeClassifier())])

In [20]:
y_pred = pipe.predict(X_test)

In [21]:
y_pred

array([0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 0], dtype=int64)

In [22]:
y_test

205    0
696    0
596    1
770    0
20     0
      ..
328    1
168    0
746    0
223    0
379    0
Name: Survived, Length: 179, dtype: int64

## Accuracy of model

In [23]:
from sklearn.metrics import accuracy_score

In [24]:
score = accuracy_score(y_test, y_pred)
score

0.8100558659217877

In [25]:
# pipe.score(X_test, y_test)

0.8100558659217877

## Exploring the Pipeline

In [26]:
pipe.named_steps

{'imputing': ColumnTransformer(remainder='passthrough',
                   transformers=[('age_imputer', SimpleImputer(), [2]),
                                 ('embarked_imputer',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'encoding': ColumnTransformer(remainder='passthrough',
                   transformers=[('embarked_sex_ohe',
                                  OneHotEncoder(sparse_output=False), [1, 3])]),
 'scaling': ColumnTransformer(remainder='passthrough',
                   transformers=[('scale_all', MinMaxScaler(),
                                  slice(0, 10, None))]),
 'selection': SelectKBest(k=8, score_func=<function chi2 at 0x0000025484F7FEC0>),
 'model': DecisionTreeClassifier()}

In [27]:
# imputed mean Age by the imputing transformer
pipe.named_steps['imputing'].transformers_[0][1].statistics_

array([29.48755712])

In [28]:
# imputed most frequest Embarked value by imputing transformer
pipe.named_steps['imputing'].transformers_[1][1].statistics_

array(['S'], dtype=object)

In [29]:
# output columns after OneHotEncoding of Sex and Embarked columns
pipe.named_steps['encoding'].transformers_[0][1].get_feature_names_out()

array(['x0_C', 'x0_Q', 'x0_S', 'x1_female', 'x1_male'], dtype=object)

## Cross-validation using Pipeline

In [30]:
from sklearn.model_selection import cross_val_score

In [31]:
score = cross_val_score(pipe, X_train, y_train, scoring = 'accuracy', cv=5)

In [32]:
score

array([0.79020979, 0.7972028 , 0.73943662, 0.84507042, 0.78873239])

In [33]:
score.mean()

0.7921304048064611

## GridSearch using Pipeline

In [34]:
from sklearn.model_selection import GridSearchCV

In [44]:
# gridsearchcv
params = { 
    'model__max_depth':[1,2,3,4,5,None]
}       
                            

In [45]:
grid = GridSearchCV(pipe, params, cv=5,  scoring='accuracy')

In [46]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('imputing',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('age_imputer',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('embarked_imputer',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('encoding',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('embarked_sex_ohe',
                                                                         OneHotEncoder(sparse_output=False),
                                                                         [1,
                                                                          3])])),
                                       ('scaling',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale_all',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('selection',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x0000025484F7FEC0>)),
                                       ('model', DecisionTreeClassifier())]),
             param_grid={'model__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [47]:
grid.best_score_

0.7963557569191372

In [48]:
grid.best_params_

{'model__max_depth': 3}

## Exporting the Pipeline


In [49]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))